# 📊 LocalTrack Experiment Analysis & Multi-Run Comparison

Query historical experiment runs, fetch downsampled time-series loss curves via LTTB, and analyze hyperparameter correlations.

In [ ]:
import pandas as pd
import requests

BASE_URL = "http://127.0.0.1:8000"

# Fetch all projects
projects = requests.get(f"{BASE_URL}/api/localtrack/projects").json()
print("Available Projects:", [p["name"] for p in projects["projects"]])

## 1. Fetch Runs & Compare Summary Metrics

In [ ]:
# List all runs in project
runs_resp = requests.get(f"{BASE_URL}/api/localtrack/runs", params={"project_id": "horrible-sft"}).json()
runs = runs_resp.get("runs", [])

df_runs = pd.DataFrame([
    {
        "id": r["id"],
        "name": r["name"],
        "status": r["status"],
        "lr": r["config"].get("learning_rate"),
        "final_loss": r["summary"].get("train/loss"),
        "eval_acc": r["summary"].get("eval/accuracy"),
        "tags": ", ".join(r["tags"]),
    }
    for r in runs
])

df_runs.sort_values(by="eval_acc", ascending=False) if not df_runs.empty else df_runs

## 2. Query Downsampled Metric Time-Series (LTTB)

In [ ]:
if len(runs) > 0:
    run_ids = [r["id"] for r in runs[:3]]
    query_payload = {
        "run_ids": run_ids,
        "keys": ["train/loss", "eval/accuracy"],
        "max_points": 100,
        "smoothing": 0.3,
    }
    metrics_resp = requests.post(f"{BASE_URL}/api/localtrack/metrics/query", json=query_payload).json()
    print(f"Retrieved {len(metrics_resp['series'])} metric series.")
    for s in metrics_resp['series']:
        print(f"Run {s['run_id']} | Key: {s['key']} | Points: {len(s['values'])} | Min: {min(s['values']):.4f}")